# Quantitative Market Dataset Builder

This notebook is designed to maximize data volume for Deep Learning models (like LSTM or Temporal Fusion Transformers). It downloads historical market data for an expanded universe of 20 assets across multiple uncorrelated markets, stretching back to 1990.

**Key Features:**
* Fetches historical data using the `yfinance` API.
* Covers broad US Equities, Global Equities, US Sectors, Commodities, Fixed Income, and Cryptocurrencies.
* Calculates essential financial features automatically (e.g., Log Returns, Volatility).

In [1]:
# Install required libraries for data fetching, manipulation, and visualization
!pip install yfinance pandas numpy matplotlib

import yfinance as yf
import pandas as pd
import numpy as np


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Data Fetching & Feature Engineering Function

The following function iterates through a defined dictionary of assets, downloads the maximum available historical data, and calculates foundational quantitative features:

* **Log-Returns**: Used for their time-additive properties. 
  $r_t = \ln \left( \frac{P_t}{P_{t-1}} \right)$
* **Simple Returns**: Used as a standard baseline comparison.
* **Historical Volatility (20-day annualized)**: Measures the dispersion of returns for a given security.
  $HV_{20} = \sigma_{20} \cdot \sqrt{252}$ 

Missing values are dropped to ensure clean inputs for downstream machine learning models.

In [2]:
def fetch_and_prep_multi_market(tickers, start_date="1990-01-01", end_date="2026-03-24"):
    """
    Downloads data for a massive list of tickers, calculates features independently,
    and returns a combined master DataFrame.
    """
    all_dataframes = []
    
    for category, ticker_list in tickers.items():
        print(f"\n--- Downloading category: {category} ---")
        for ticker in ticker_list:
            print(f"Processing {ticker}...")
            try:
                # 1. Download data (history maxed out to start_date)
                df = yf.download(ticker, start=start_date, end=end_date, progress=False)
                
                if df.empty:
                    print(f"  -> Warning: No data found for {ticker}. Skipping...")
                    continue
                    
                # Flatten MultiIndex columns if present (common in recent yfinance updates)
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.droplevel(1)
                    
                # 2. Add structural columns
                df['ticker'] = ticker
                df['asset_class'] = category # Useful static covariate for advanced models like TFT
                
                # 3. Calculate Financial Features 
                # Log-Returns
                df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
                
                # Simple returns for baseline comparison
                df['simple_return'] = df['Close'].pct_change()
                
                # Historical Volatility (20-day annualized)
                window_size = 20
                df['historical_volatility_20d'] = df['log_return'].rolling(window=window_size).std() * np.sqrt(252)
                
                # 4. Clean Missing Values (drops NaNs generated by rolling windows and shifts)
                df = df.dropna()
                
                # 5. Reorder columns for readability
                cols = ['ticker', 'asset_class', 'Open', 'High', 'Low', 'Close', 'Volume', 'simple_return', 'log_return', 'historical_volatility_20d']
                df = df[cols]
                
                all_dataframes.append(df)
                print(f"  -> {ticker}: {len(df)} records processed.")
                
            except Exception as e:
                print(f"  -> Error processing {ticker}: {e}")
            
    # 6. Combine all individual dataframes into a master dataframe
    print("\nMerging all assets into the Expanded Master Dataset...")
    master_df = pd.concat(all_dataframes)
    master_df = master_df.sort_index()
    
    return master_df

### Define the Asset Universe

We categorize the assets into distinct classes. This structure is highly beneficial for deep learning models that can leverage static categorical variables to learn distinct market behaviors.

In [3]:
# Expanded Universe of Assets (20 Tickers across 6 Asset Classes)
expanded_universe = {
    "Equities_US_Broad": ["SPY", "QQQ", "DIA", "IWM"],  # S&P500, Nasdaq, Dow Jones, Russell 2000
    "Equities_Global": ["EFA", "EEM", "URTH"],          # Developed ex-US, Emerging Markets, MSCI World
    "Sectors_US": ["XLF", "XLV", "XLE", "XLK"],         # Financials, Health, Energy, Tech
    "Commodities": ["GLD", "SLV", "USO", "DBA"],        # Gold, Silver, Oil, Agriculture
    "Fixed_Income": ["TLT", "AGG"],                     # 20+ Year Treasuries, Broad Bonds
    "Cryptocurrency": ["BTC-USD", "ETH-USD", "BNB-USD"] # Bitcoin, Ethereum, Binance Coin
}

### Execution and Export

Finally, we trigger the ingestion process, print the dataset statistics to verify the data balance, and save the final quantitative master dataset to a CSV file.

In [4]:
# Execute the pipeline
master_market_df = fetch_and_prep_multi_market(tickers=expanded_universe)

# Display Summary Statistics
print("\n--- Quantitative Master Dataset Statistics ---")
print(f"Total global records: {len(master_market_df)}")
print("\nBreakdown by Asset Class:")
print(master_market_df['asset_class'].value_counts())

# Save the master dataset
output_filename = "../data/quantitative_market_dataset.csv"
master_market_df.to_csv(output_filename)
print(f"\nHigh-volume dataset successfully saved to {output_filename}!")


--- Downloading category: Equities_US_Broad ---
Processing SPY...


C:\Users\frang\AppData\Local\Temp\ipykernel_34468\3797810990.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)


  -> SPY: 8323 records processed.
Processing QQQ...


C:\Users\frang\AppData\Local\Temp\ipykernel_34468\3797810990.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)


  -> QQQ: 6781 records processed.
Processing DIA...


...

  -> BTC-USD: 4186 records processed.
Processing ETH-USD...


C:\Users\frang\AppData\Local\Temp\ipykernel_34468\3797810990.py:33: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['simple_return'] = df['Close'].pct_change()
C:\Users\frang\AppData\Local\Temp\ipykernel_34468\3797810990.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)


  -> ETH-USD: 3037 records processed.
Processing BNB-USD...


C:\Users\frang\AppData\Local\Temp\ipykernel_34468\3797810990.py:33: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['simple_return'] = df['Close'].pct_change()


  -> BNB-USD: 3037 records processed.

Merging all assets into the Expanded Master Dataset...

--- Quantitative Master Dataset Statistics ---
Total global records: 113406

Breakdown by Asset Class:
asset_class
Equities_US_Broad    28644
Sectors_US           27332
Commodities          20146
Equities_Global      15458
Fixed_Income         11566
Cryptocurrency       10260
Name: count, dtype: int64

High-volume dataset successfully saved to ../data/quantitative_market_dataset.csv!
